In [0]:
from pyspark.sql.types import StructType, StructField, StringType

path = "/Volumes/cms/bronze/raw_files/"
source_file = "Medicare_Part_D_Prescribers_2024.csv"

columns = [
    "Prscrbr_NPI", "Prscrbr_Last_Org_Name", "Prscrbr_First_Name", "Prscrbr_City",
    "Prscrbr_State_Abrvtn", "Prscrbr_State_FIPS", "Prscrbr_Type", "Prscrbr_Type_Src",
    "Brnd_Name", "Gnrc_Name", "Tot_Clms", "Tot_30day_Fills", "Tot_Day_Suply",
    "Tot_Drug_Cst", "Tot_Benes", "GE65_Sprsn_Flag", "GE65_Tot_Clms",
    "GE65_Tot_30day_Fills", "GE65_Tot_Drug_Cst", "GE65_Tot_Day_Suply",
    "GE65_Bene_Sprsn_Flag", "GE65_Tot_Benes",
]

# All 22 source columns as STRING, plus a column to catch malformed rows
schema = StructType(
    [StructField(c, StringType(), True) for c in columns]
    + [StructField("_corrupt_record", StringType(), True)]
)
print(len(columns), "source columns")

In [0]:
header_line = dbutils.fs.head(path + source_file, 2000).splitlines()[0]
file_columns = header_line.split(",")

assert file_columns == columns, f"Header mismatch:\n{set(file_columns) ^ set(columns)}"
print("Header matches expected columns.")

In [0]:
from pyspark.sql import functions as F

raw_df = (
    spark.read
    .option("header", True)
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .schema(schema)
    .csv(path + source_file)
)

bronze_df = (
    raw_df
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn(
        "data_year",
        F.regexp_extract(F.col("_metadata.file_path"), r"_(\d{4})\.csv$", 1).cast("int"),
    )
)
# How to read tricky rows. Some drug names contain commas inside quotes. The instructions say “commas inside quotes are part of the name, not a column break,” so those rows don’t get split wrong.

# What to do with a broken row. Instead of crashing or quietly throwing the row away, set it aside in a separate column so we can count it and look at it.

# Three labels added to every row: which file it came from, when it was loaded, and what year the data covers (taken from the file name, which is why we added “2024” to it).

In [0]:
display(bronze_df.limit(10))

In [0]:
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("cms.bronze.partd_prescriber_drug")
)

# bronze_df is only a plan: read this file, with this schema, add these three columns. Nothing exists in memory until something forces the plan to run.

In [0]:
%sql
SELECT data_year, COUNT(*) AS row_count
FROM cms.bronze.partd_prescriber_drug
GROUP BY data_year;

-- We are limited to 2024 due to the Free Edition of Databricks
-- If you have a Databricks Pro or Databricks Premium workspace, you can run this query for all years.



In [0]:
%sql
DESCRIBE DETAIL cms.bronze.partd_prescriber_drug;